# 09 · Convolutions, By Hand

The lesson argues that a convolution is an *assumption* rather than a trick: nearby
pixels are related, and a pattern means the same thing wherever it appears. This
writes the arithmetic out so that assumption is visible rather than asserted.

No libraries. The sliding window is the whole point, and calling a library function
would hide exactly the part worth seeing. Everything is a nested list and a loop,
which is slow and legible — the right trade for sixteen by sixteen pixels.

In [1]:
import math, random
random.seed(9)

# A 16x16 greyscale image, built rather than loaded so the notebook needs nothing.
# A light square on a dark ground, with a little noise so nothing is suspiciously clean.
H = W = 16
img = [[0.1 + random.gauss(0, 0.02) for _ in range(W)] for _ in range(H)]
for y in range(4, 12):
    for x in range(5, 13):
        img[y][x] = 0.85 + random.gauss(0, 0.02)

SHADES = " .:-=+*#%@"
def show(grid, label=""):
    lo = min(min(r) for r in grid); hi = max(max(r) for r in grid)
    rng = (hi - lo) or 1.0
    print(label)
    for row in grid:
        print("   " + "".join(SHADES[min(9, int((v - lo) / rng * 9.999))] for v in row))
    print(f"   range {lo:+.2f} to {hi:+.2f}\n")

show(img, "the image (light square on dark ground)")

the image (light square on dark ground)
                   
                   
      .            
                   
        @@@@@@@@   
        @@@@@@@@   
        @@@@@@@@.  
        @@@@@%@@   
        @@@@%@@@   
        @@@@@@@@   
        @@@@@%@@   
        @@@@@@@@.  
                   
     .   .         
   .               
      .        .   
   range +0.05 to +0.89



## One operation, three behaviours

The kernel is nine numbers. Change the nine numbers and the same code detects vertical
edges, horizontal edges, or blurs — which is the point that gets lost when convolution
is introduced as "a filter". It is not a filter for anything in particular. It is a
patch-matcher, and what it matches is whatever you put in the window.

In a trained network nobody chooses these numbers. They are learned, and the early
layers of real CNNs reliably converge on something close to the edge detectors below.

In [2]:
def convolve(grid, kernel):
    """Valid convolution, written out. No library — the sliding is the lesson."""
    kh, kw = len(kernel), len(kernel[0])
    out = []
    for y in range(len(grid) - kh + 1):
        row = []
        for x in range(len(grid[0]) - kw + 1):
            total = 0.0
            for ky in range(kh):
                for kx in range(kw):
                    total += grid[y + ky][x + kx] * kernel[ky][kx]
            row.append(total)
        out.append(row)
    return out

VERTICAL   = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]     # Sobel: left-right change
HORIZONTAL = [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]     # Sobel: top-bottom change
BLUR       = [[1/9]*3 for _ in range(3)]

show(convolve(img, VERTICAL),   "vertical edges — the left and right sides of the square")
show(convolve(img, HORIZONTAL), "horizontal edges — the top and bottom")
show(convolve(img, BLUR),       "blur — same machinery, different nine numbers")

vertical edges — the left and right sides of the square
   =++==++====+=+
   ++===++====+==
   ++=**++===+--=
   =+=%%=+===+..=
   =++@@++===+  =
   =++@@+++===  =
   +++@@++===+  +
   ++=@@++==++  +
   ++=@@+===++  +
   ===@@=++==+  =
   ==+%%++===+..=
   +==**+==++=--=
   +===++=+=+==+=
   =+==++=++===+=
   range -3.09 to +3.08

horizontal edges — the top and bottom
   ++====+++=====
   +++++====++===
   ===*%@@@@@@%*+
   ===*%@@@@@@%*+
   ++++=++++++=++
   +==+=========+
   =+==+++==+++==
   +===++=++=====
   ++=++++++==+++
   +++-.      .-+
   +++-.      .-+
   ++++==+++=====
   ==+++===++====
   range -3.07 to +3.09

blur — same machinery, different nine numbers
                 
                 
      .:------:. 
      :=******=: 
      -*@@@@@@*- 
      -*@@@@@@*- 
      -*@@@@@@*- 
      -*@@@@@@*- 
      -*@@@@@@*- 
      -*@@@@@@*- 
      :=******=: 
      .:------:. 
                 
                 
   range +0.08 to +0.86



## What weight sharing actually buys

Move the square. Run the same kernel. Nothing is retrained and no parameter changes.

In [3]:
# The same kernel, the same square, moved. Weight sharing means no relearning.
moved = [[0.1 + random.gauss(0, 0.02) for _ in range(W)] for _ in range(H)]
for y in range(1, 8):
    for x in range(2, 9):
        moved[y][x] = 0.85 + random.gauss(0, 0.02)

a = convolve(img, VERTICAL)
b = convolve(moved, VERTICAL)
peak = lambda g: max(max(abs(v) for v in row) for row in g)
print(f"strongest vertical-edge response, original position : {peak(a):.2f}")
print(f"strongest vertical-edge response, square moved      : {peak(b):.2f}")
print("\nsame nine weights, no retraining, no new parameters — the detector moved with it.")
print("a dense layer would have learned 'edge at column 12' and known nothing about column 8.")

strongest vertical-edge response, original position : 3.09
strongest vertical-edge response, square moved      : 3.10

same nine weights, no retraining, no new parameters — the detector moved with it.
a dense layer would have learned 'edge at column 12' and known nothing about column 8.


This is translation equivariance, and it is the assumption doing the work. A dense
layer would have to see the square in every position during training to respond to it
in every position. The convolution never learns a position at all.

## The parameter count, which is the headline argument

In [4]:
def dense_params(h, w, channels, units):
    return h * w * channels * units + units

def conv_params(k, in_ch, out_ch):
    return k * k * in_ch * out_ch + out_ch

print(f"{'layer':<46}{'parameters':>14}")
print("-" * 60)
print(f"{'dense: 224x224x3 image -> 1000 units':<46}{dense_params(224,224,3,1000):>14,}")
print(f"{'conv:  3x3 kernel, 3 in -> 64 filters':<46}{conv_params(3,3,64):>14,}")
print(f"{'conv:  3x3 kernel, 64 in -> 128 filters':<46}{conv_params(3,64,128):>14,}")

ratio = dense_params(224,224,3,1000) / conv_params(3,3,64)
print(f"\nthe first conv layer has {ratio:,.0f}x fewer parameters than the dense one")
print("and unlike the dense layer it works on an image of any size,")
print("because the parameters belong to the kernel rather than to the pixels")

layer                                             parameters
------------------------------------------------------------
dense: 224x224x3 image -> 1000 units             150,529,000
conv:  3x3 kernel, 3 in -> 64 filters                  1,792
conv:  3x3 kernel, 64 in -> 128 filters               73,856

the first conv layer has 84,001x fewer parameters than the dense one
and unlike the dense layer it works on an image of any size,
because the parameters belong to the kernel rather than to the pixels


Read the second consequence as carefully as the first. The dense layer's parameters
are tied to a specific input size, so a 225-pixel image needs a different layer. The
convolution's parameters belong to the kernel, so the same trained filter runs on any
image — which is why convolutional networks handle variable input sizes naturally and
dense ones do not.

## How small windows end up seeing large objects

In [5]:
# Receptive field: how much of the input one unit can see, by depth.
print(f"{'layer':>6}{'kernel':>8}{'stride':>8}{'sees (pixels)':>16}")
field, stride_prod = 1, 1
for layer, (k, s) in enumerate([(3,1),(3,1),(3,2),(3,1),(3,2),(3,1)], start=1):
    field += (k - 1) * stride_prod
    stride_prod *= s
    print(f"{layer:>6}{k:>8}{s:>8}{field:>16}")

print("\nsix layers of 3x3 and a unit at the top sees a large patch of the original.")
print("no layer ever looked at more than nine numbers at once.")

 layer  kernel  stride   sees (pixels)
     1       3       1               3
     2       3       1               5
     3       3       2               7
     4       3       1              11
     5       3       2              15
     6       3       1              23

six layers of 3x3 and a unit at the top sees a large patch of the original.
no layer ever looked at more than nine numbers at once.


The receptive field grows with depth, and faster wherever a stride halves the
resolution. This is the resolution of the apparent paradox in the lesson: a network
built entirely from three-by-three windows responds to whole objects, because a unit
high in the stack is looking at units that were themselves looking at units.

Compare this with attention, which reaches every position from the first layer and
pays quadratically for the privilege. The convolution reaches far more cheaply and
only in a fixed pattern; attention reaches anywhere and pays for it. That is the same
trade the lesson describes as a data budget, arriving as an architectural one.

## What to carry away

- A convolution is a **patch-matcher with shared weights**, not a filter for edges.
  Edges are just what these particular nine numbers happen to find.
- **Weight sharing is the assumption**, and it is what makes position irrelevant.
- The **parameter saving is enormous**, and the input-size independence that comes
  with it matters just as much in practice.
- **Depth buys reach.** Small windows stacked see large things, cheaply and in a
  fixed pattern — which is precisely what attention gives up in exchange for
  generality.